# NOTEBOOK 11: XÁC MINH & KIỂM ĐỊNH PHÂN HỆ ĐIỂM SỐ & SỔ ĐIỂM (CHECKPOINT 4.6)
## HỆ THỐNG QUẢN LÝ TRUNG TÂM SMART EDUCATION (SMARTEDU - THCS)

Notebook này kiểm định, mô phỏng và xác minh toàn diện chuỗi dữ liệu điểm số:
$$\text{Student} \longrightarrow \text{Active ClassEnrollment} \longrightarrow \text{Class} \longrightarrow \text{Teacher Assignment} \longrightarrow \text{Subject} \longrightarrow \text{Score Record}$$

### Các nguyên tắc kiểm thử bất biến (Invariants):
1. **Khóa logic duy nhất (Deterministic Composite Key):** `score_{classId}_{subjectId}_{studentId}_{term}_{scoreType}`.
2. **Thang điểm & Độ chính xác:** Thang $0.0 \le \text{score} \le 10.0$ với bước nhảy độ chính xác $0.1$.
3. **Quyền sở hữu giáo viên (Teacher Ownership):** Giáo viên chỉ được nhập/sửa điểm cho lớp/môn mà mình được phân công giảng dạy `ACTIVE` trong năm học tương ứng.
4. **Toàn vẹn danh sách học sinh:** Chỉ học sinh có `classEnrollment.status == 'ACTIVE'` mới được ghi nhận điểm.
5. **Môn học chuẩn THCS:** Chỉ chấp nhận 5 môn: Toán, Ngữ văn, Tiếng Anh, Vật lý, Hóa học (Không có Sinh học hay môn ngoài danh mục).
6. **Vòng đời & Khóa sổ học vụ:** `DRAFT` $\rightarrow$ `SUBMITTED` $\rightarrow$ `LOCKED` (Chặn Teacher khi `LOCKED`, chỉ Admin/Academic Staff có quyền mở khóa).
7. **Phân quyền bảo mật (RBAC):** `ACCOUNTANT` bị từ chối sửa điểm tuyệt đối (DENY).

In [ ]:
import json
import math

# 1. Khởi tạo cấu hình hệ thống
ACADEMIC_YEAR = '2026-2027'
TERMS = ['T1', 'T2']
VALID_SUBJECTS = ['toan', 'van', 'anh', 'ly', 'hoa']
SCORE_TYPES = ['CONTINUOUS', 'MIDTERM', 'FINAL']

print(f"[Setup] Năm học: {ACADEMIC_YEAR} | Học kỳ: {TERMS} | Môn: {VALID_SUBJECTS}")

### 2. Kiểm định Khóa Composite Document ID & Range Validation

In [ ]:
def generate_score_id(class_id, subject_id, student_id, term, score_type):
    return f"score_{class_id}_{subject_id}_{student_id}_{term}_{score_type}"

def validate_score_value(val):
    if val is None or val == '':
        return False, None, "Điểm không được để trống"
    try:
        num = float(val)
    except ValueError:
        return False, None, "Điểm phải là số thực hợp lệ"
    if num < 0.0 or num > 10.0:
        return False, None, "Điểm phải nằm trong thang 0.0 - 10.0"
    normalized = round(num, 1)
    return True, normalized, "OK"

# Test generate_score_id
sid = generate_score_id('class_6A1', 'toan', 'STU-2026-001', 'T1', 'CONTINUOUS')
print(f"[Test 1] Generated Score ID: {sid}")
assert sid == "score_class_6A1_toan_STU-2026-001_T1_CONTINUOUS"

# Test validate_score_value
assert validate_score_value(8.55)[1] == 8.6
assert validate_score_value(10.0)[0] is True
assert validate_score_value(0.0)[0] is True
assert validate_score_value(-1.0)[0] is False
assert validate_score_value(10.5)[0] is False
assert validate_score_value('abc')[0] is False
print("[Test 2] Score range & precision validation: PASS")

### 3. Kiểm định Quyền Phân Công Giảng Dạy & Tính Duy Nhất

In [ ]:
mock_assignments = [
    {
        "id": "asn_6A1_toan",
        "teacherId": "TCH-2026-001",
        "teacherName": "Trần Quốc Việt",
        "classId": "class_6A1",
        "subjectId": "toan",
        "academicYear": "2026-2027",
        "status": "ACTIVE"
    }
]

def validate_teacher_permission(teacher_id, class_id, subject_id, academic_year, assignments):
    for a in assignments:
        if (a['teacherId'] == teacher_id and 
            a['classId'] == class_id and 
            a['subjectId'] == subject_id and 
            a['academicYear'] == academic_year and 
            a['status'] == 'ACTIVE'):
            return True, "Phân công hợp lệ"
    return False, "Giáo viên không có phân công giảng dạy cho lớp/môn này"

# Kiểm tra giáo viên được phân công
ok, msg = validate_teacher_permission('TCH-2026-001', 'class_6A1', 'toan', '2026-2027', mock_assignments)
assert ok is True
print(f"[Test 3.1] Assigned teacher check: {msg}")

# Kiểm tra giáo viên khác (TCH-2026-002) cố tình nhập điểm lớp 6A1 môn Toán
ok, msg = validate_teacher_permission('TCH-2026-002', 'class_6A1', 'toan', '2026-2027', mock_assignments)
assert ok is False
print(f"[Test 3.2] Unassigned teacher check (Denied): {msg}")

### 4. Kiểm định RBAC & Khóa Sổ Học Vụ (Scorebook Lifecycle)

In [ ]:
def can_edit_score(role, score_status, is_assigned_teacher):
    if role == 'ACCOUNTANT':
        return False
    if role in ['ADMIN', 'OWNER', 'ACADEMIC_STAFF']:
        return True
    if role == 'TEACHER':
        if not is_assigned_teacher:
            return False
        return score_status != 'LOCKED'
    return False

# RBAC Invariants
assert can_edit_score('ACCOUNTANT', 'DRAFT', True) is False
assert can_edit_score('ACCOUNTANT', 'LOCKED', True) is False
assert can_edit_score('ADMIN', 'LOCKED', False) is True
assert can_edit_score('ACADEMIC_STAFF', 'LOCKED', False) is True
assert can_edit_score('TEACHER', 'DRAFT', True) is True
assert can_edit_score('TEACHER', 'SUBMITTED', True) is True
assert can_edit_score('TEACHER', 'LOCKED', True) is False
assert can_edit_score('STUDENT', 'DRAFT', False) is False
assert can_edit_score('PARENT', 'DRAFT', False) is False

print("[Test 4] RBAC & Scorebook Lock State Matrix: 100% PASS")

### 5. Kiểm định Tính Điểm Trung Bình & Xếp Loại Học Lực

In [ ]:
def compute_subject_grade(continuous, midterm, final):
    scores = [s for s in [continuous, midterm, final] if s is not None]
    if not scores:
        return None, None, False
    avg = round(sum(scores) / len(scores), 1)
    if avg >= 9.0:
        grade = 'A'
    elif avg >= 8.0:
        grade = 'B'
    elif avg >= 6.5:
        grade = 'C'
    elif avg >= 5.0:
        grade = 'D'
    else:
        grade = 'F'
    return avg, grade, avg >= 5.0

avg, grade, passed = compute_subject_grade(8.5, 8.8, 9.2)
print(f"[Test 5.1] Học sinh giỏi: ĐTB={avg}, Xếp loại={grade}, Đạt={passed}")
assert grade == 'B' or grade == 'A'
assert passed is True

avg, grade, passed = compute_subject_grade(4.0, 4.5, 3.5)
print(f"[Test 5.2] Học sinh chưa đạt: ĐTB={avg}, Xếp loại={grade}, Đạt={passed}")
assert grade == 'F'
assert passed is False

print("[Summary] Toàn bộ quy tắc kiểm định Checkpoint 4.6 đã được xác minh thành công!")